### Support Vector Machine — Does This Patient Have Heart Disease?

A **Support Vector Machine (SVM)** separates two classes by drawing the dividing line that leaves the **widest empty street** between them. The points on the edge of that street — the **support vectors** — are the only ones that shape the line.

Here we'll use the classic **Cleveland Heart Disease** dataset: predict whether a patient has heart disease from 13 clinical measurements (age, blood pressure, cholesterol, ECG results, and so on).

If you also worked through the **credit-default** notebook, watch the difference: there the classes were badly imbalanced (78/22) and the model missed most defaulters. Here the classes are **nearly balanced (~54/46)** — and you'll see how much healthier the recall looks as a result.

### What the columns mean

All 13 features are clinical measurements. Several are **codes, not quantities**:

- `age` — age in years
- `sex` — **1 = male, 0 = female**
- `cp` — chest pain type: **1 = typical angina, 2 = atypical angina, 3 = non-anginal pain, 4 = asymptomatic**
- `trestbps` — resting blood pressure (mm Hg)
- `chol` — serum cholesterol (mg/dl)
- `fbs` — fasting blood sugar > 120 mg/dl (**1 = true, 0 = false**)
- `restecg` — resting ECG: **0 = normal, 1 = ST-T abnormality, 2 = left-ventricular hypertrophy**
- `thalach` — maximum heart rate achieved
- `exang` — exercise-induced angina (**1 = yes, 0 = no**)
- `oldpeak` — ST depression induced by exercise
- `slope` — slope of the peak exercise ST segment: **1 = upsloping, 2 = flat, 3 = downsloping**
- `ca` — number of major vessels (0–3) colored by fluoroscopy
- `thal` — **3 = normal, 6 = fixed defect, 7 = reversible defect**
- `num` — **the raw diagnosis: 0 = no disease, 1–4 = increasing severity of disease**

In [4]:
import os
print(os.getcwd())

C:\Users\Shaik\ML-Foundations\Classification\SVM(SupportVectorMachine)


In [1]:
import pandas as pd

In [13]:
df= pd.read_csv("../../Datasets/heart_disease.csv")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [14]:

df['thal'].unique()

array(['6.0', '3.0', '7.0', '?'], dtype=object)

In [15]:
df= pd.read_csv("../../Datasets/heart_disease.csv", na_values= '?')
print('rows, columns:', df.shape)
df.head()

rows, columns: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [18]:
#we have some null_values
df.isnull().sum()

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
num         0
dtype: int64

In [19]:
df.shape

(303, 14)

In [20]:
#we are droping null_values compare to number rows it is not that big of a deal
df= df.dropna()
df.shape

(297, 14)